In [86]:
#import
from pathlib import Path
from urllib.request import urlretrieve
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pandas as pd
import rdata
import numpy as np

In [87]:

# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================

# URL base del repositorio oficial donde están los archivos .rda
BASE_URL = "https://github.com/paezha/idealista18/raw/master/data"

# Carpeta local donde guardaremos los archivos descargados
DATA_DIR = Path("idealista18_data")
DATA_DIR.mkdir(exist_ok=True)

# Relación entre ciudad y archivo de ventas correspondiente
SALE_FILES = {
    "Barcelona": "Barcelona_Sale.rda",
    "Madrid": "Madrid_Sale.rda",
    "Valencia": "Valencia_Sale.rda",
}


# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def download_file(filename: str) -> Path:
    """
    Descarga un archivo desde el repositorio oficial si no existe en local.

    Parameters
    ----------
    filename : str
        Nombre del archivo a descargar, por ejemplo 'Madrid_Sale.rda'.

    Returns
    -------
    Path
        Ruta local al archivo descargado.
    """
    local_path = DATA_DIR / filename

    if not local_path.exists():
        url = f"{BASE_URL}/{filename}"
        print(f"Descargando {filename}...")
        urlretrieve(url, local_path)

    return local_path


def load_rda(filename: str) -> dict:
    """
    Carga un archivo .rda y devuelve su contenido como diccionario.

    Nota:
    Un archivo .rda puede contener uno o varios objetos de R.
    Esta librería normalmente los devuelve en un diccionario
    donde la clave es el nombre del objeto almacenado.

    Parameters
    ----------
    filename : str
        Nombre del archivo .rda.

    Returns
    -------
    dict
        Diccionario con los objetos cargados desde el archivo.
    """
    local_path = download_file(filename)
    data = rdata.read_rda(str(local_path), default_encoding="utf8")
    return data


def load_city_sales(city_name: str, filename: str) -> pd.DataFrame:
    """
    Carga el dataset de ventas de una ciudad y añade una columna CITY.

    Parameters
    ----------
    city_name : str
        Nombre de la ciudad, por ejemplo 'Madrid'.
    filename : str
        Nombre del archivo .rda asociado.

    Returns
    -------
    pd.DataFrame
        DataFrame con los datos de ventas de esa ciudad.
    """
    rda_content = load_rda(filename)

    # El objeto dentro del .rda suele llamarse igual que el archivo sin extensión
    object_name = filename.replace(".rda", "")

    if object_name not in rda_content:
        raise KeyError(
            f"No se encontró el objeto '{object_name}' dentro de '{filename}'. "
            f"Objetos disponibles: {list(rda_content.keys())}"
        )

    df = rda_content[object_name].copy()

    # Añadimos una columna para que el modelo sepa de qué ciudad viene cada vivienda
    df["CITY"] = city_name

    return df


def load_all_sales() -> pd.DataFrame:
    """
    Carga y concatena los datasets de ventas de Barcelona, Madrid y Valencia.

    Returns
    -------
    pd.DataFrame
        DataFrame unificado con las 3 ciudades.
    """
    city_dfs = []

    for city_name, filename in SALE_FILES.items():
        print(f"Cargando datos de {city_name}...")
        city_df = load_city_sales(city_name, filename)
        city_dfs.append(city_df)

    df_all = pd.concat(city_dfs, ignore_index=True)
    return df_all


# ============================================================
# CARGA PRINCIPAL
# ============================================================

# Cargar las tres ciudades en un único DataFrame
df = load_all_sales()


# ============================================================
# INSPECCIÓN BÁSICA
# ============================================================

print("\nPrimeras filas:")
print(df.head())

print("\nDimensiones del dataset unificado:")
print(df.shape)

print("\nCiudades incluidas:")
print(df["CITY"].value_counts())

print("\nColumnas disponibles:")
print(df.columns.tolist())

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos por columna:")
print(df.isnull().sum().sort_values(ascending=False).head(20))

Cargando datos de Barcelona...


c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "bbox". The underlying R object is returned instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "crs". The underlying R object is returned instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "XY". The constructor for class "POINT" will be used instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "POINT". The constructor for class "sfg" will be used instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversi

Cargando datos de Madrid...


c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "bbox". The underlying R object is returned instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "crs". The underlying R object is returned instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "XY". The constructor for class "POINT" will be used instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "POINT". The constructor for class "sfg" will be used instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversi

Cargando datos de Valencia...


c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "bbox". The underlying R object is returned instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "crs". The underlying R object is returned instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "XY". The constructor for class "POINT" will be used instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversion\_conversion.py:900: UserWarning: Missing constructor for R class "POINT". The constructor for class "sfg" will be used instead.
  warnings.warn(
c:\Users\Eddy\AppData\Local\Programs\Python\Python313\Lib\site-packages\rdata\conversi


Primeras filas:
                 ASSETID  PERIOD     PRICE    UNITPRICE  CONSTRUCTEDAREA  \
0  A11898131848556022319  201803  323000.0  3845.238095               84   
1  A18099432772155664747  201803  217000.0  2583.333333               84   
2   A2003099089407882787  201803  114000.0  1407.407407               81   
3   A1010373782315301134  201803  378000.0  4784.810127               79   
4  A12978912200216838006  201803  434000.0  3909.909910              111   

   ROOMNUMBER  BATHNUMBER  HASTERRACE  HASLIFT  HASAIRCONDITIONING  ...  \
0           4           1           1        1                   1  ...   
1           3           2           0        1                   1  ...   
2           2           1           0        1                   1  ...   
3           2           1           0        1                   0  ...   
4           4           2           1        1                   1  ...   

   BUILTTYPEID_3  DISTANCE_TO_CITY_CENTER  DISTANCE_TO_METRO  \
0          

In [88]:
df["CITY"].value_counts()


CITY
Madrid       94815
Barcelona    61486
Valencia     33622
Name: count, dtype: int64

In [89]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 189923 entries, 0 to 189922
Data columns (total 45 columns):
 #   Column                         Non-Null Count   Dtype  
---  ------                         --------------   -----  
 0   ASSETID                        189923 non-null  object 
 1   PERIOD                         189923 non-null  Int32  
 2   PRICE                          189923 non-null  float64
 3   UNITPRICE                      189923 non-null  float64
 4   CONSTRUCTEDAREA                189923 non-null  Int32  
 5   ROOMNUMBER                     189923 non-null  Int32  
 6   BATHNUMBER                     189923 non-null  Int32  
 7   HASTERRACE                     189923 non-null  Int32  
 8   HASLIFT                        189923 non-null  Int32  
 9   HASAIRCONDITIONING             189923 non-null  Int32  
 10  AMENITYID                      189923 non-null  Int32  
 11  HASPARKINGSPACE                189923 non-null  Int32  
 12  ISPARKINGSPACEINCLUDEDINPRICE 

In [90]:
# Esta es la variable que queremos predecir con el modelo.
target = "PRICE"

# 2. Definir las variables predictoras (features)
features = [
    "CONSTRUCTEDAREA",
    "ROOMNUMBER",
    "BATHNUMBER",
    "HASTERRACE",
    "HASLIFT",
    "HASAIRCONDITIONING",
    "HASPARKINGSPACE",
    "HASBOXROOM",
    "HASWARDROBE",
    "HASSWIMMINGPOOL",
    "HASGARDEN",
    "ISDUPLEX",
    "ISSTUDIO",
    "ISINTOPFLOOR",
    "LATITUDE",
    "LONGITUDE",
    "DISTANCE_TO_CITY_CENTER",
    "DISTANCE_TO_METRO",
    "CITY",
]

# 3. Crear un nuevo DataFrame solo con las columnas necesarias
df_model = df[features + [target]].copy()

# 4. Revisar valores nulos
#print("Valores nulos por columna:")
#print(df_model.isnull().sum().sort_values(ascending=False))
df_model = df_model.dropna()
# 5. Convertir la variable CITY en variables numéricas  CITY = Madrid / Barcelona / Valencia
# CITY = Madrid / Barcelona / Valencia
# pasa a:
# CITY_Madrid, CITY_Valencia
# drop_first=True evita redundancia entre columnas.
df_model = pd.get_dummies(df_model, columns=["CITY"], drop_first=True)

# 6. Separar variables predictoras (X) y variable objetivo (y)
X = df_model.drop(columns=[target])
y = df_model[target]

# Forzar nombres de columnas a str nativo de Python
X.columns = [str(col) for col in X.columns]
# 6. Dividir en conjunto de entrenamiento y prueba
# train: datos que usa el modelo para aprender
# test: datos que usamos para evaluar qué tan bien funciona
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Volver a forzar por seguridad
X_train.columns = [str(col) for col in X_train.columns]
X_test.columns = [str(col) for col in X_test.columns]


print("\nShape final de X:", X.shape)
print("Shape de y:", y.shape)

print("\nColumnas finales usadas por el modelo:")
print(X.columns.tolist())

print("\nPrimeras filas del dataset preparado:")
print(df_model.head())



Shape final de X: (189923, 20)
Shape de y: (189923,)

Columnas finales usadas por el modelo:
['CONSTRUCTEDAREA', 'ROOMNUMBER', 'BATHNUMBER', 'HASTERRACE', 'HASLIFT', 'HASAIRCONDITIONING', 'HASPARKINGSPACE', 'HASBOXROOM', 'HASWARDROBE', 'HASSWIMMINGPOOL', 'HASGARDEN', 'ISDUPLEX', 'ISSTUDIO', 'ISINTOPFLOOR', 'LATITUDE', 'LONGITUDE', 'DISTANCE_TO_CITY_CENTER', 'DISTANCE_TO_METRO', 'CITY_Madrid', 'CITY_Valencia']

Primeras filas del dataset preparado:
   CONSTRUCTEDAREA  ROOMNUMBER  BATHNUMBER  HASTERRACE  HASLIFT  \
0               84           4           1           1        1   
1               84           3           2           0        1   
2               81           2           1           0        1   
3               79           2           1           0        1   
4              111           4           2           1        1   

   HASAIRCONDITIONING  HASPARKINGSPACE  HASBOXROOM  HASWARDROBE  \
0                   1                0           0            0   
1         

In [91]:
#Crear el modelo

# RandomForestRegressor como modelo
#
# - Funciona bien con relaciones no lineales
# - Maneja bien mezclas de variables numéricas y binarias
# - No necesita escalado de variables
model = RandomForestRegressor(
    n_estimators=100,    
    random_state=42,    
    n_jobs=-1             
)

# Entrenar el modelo
# Aquí el modelo aprende la relación entre las variables X_train
# y el precio y_train.
model.fit(X_train, y_train)

# ------------------------------------------------------------
# 3. Hacer predicciones
# ------------------------------------------------------------
# El modelo predice precios para:
# - entrenamiento (para comparar si está memorizando demasiado)
# - prueba (para medir rendimiento real)
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Métricas en entrenamiento
mae_train = mean_absolute_error(y_train, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
r2_train = r2_score(y_train, y_train_pred)

# Métricas en prueba
mae_test = mean_absolute_error(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2_test = r2_score(y_test, y_test_pred)

print("=== TRAIN ===")
print(f"MAE:  {mae_train:,.2f}")
print(f"RMSE: {rmse_train:,.2f}")
print(f"R²:   {r2_train:.4f}")

print("\n=== TEST ===")
print(f"MAE:  {mae_test:,.2f}")
print(f"RMSE: {rmse_test:,.2f}")
print(f"R²:   {r2_test:.4f}")

feature_importance = pd.Series(model.feature_importances_, index=X_train.columns)
feature_importance = feature_importance.sort_values(ascending=False)

print(feature_importance)


=== TRAIN ===
MAE:  15,224.19
RMSE: 32,017.65
R²:   0.9917

=== TEST ===
MAE:  40,568.95
RMSE: 82,447.59
R²:   0.9449
CONSTRUCTEDAREA            0.702895
LATITUDE                   0.134950
LONGITUDE                  0.061572
DISTANCE_TO_CITY_CENTER    0.039344
BATHNUMBER                 0.014499
DISTANCE_TO_METRO          0.014254
ROOMNUMBER                 0.008808
HASLIFT                    0.006258
HASSWIMMINGPOOL            0.003899
HASAIRCONDITIONING         0.002819
HASPARKINGSPACE            0.002239
HASTERRACE                 0.002055
HASWARDROBE                0.001784
HASBOXROOM                 0.001593
ISDUPLEX                   0.001072
HASGARDEN                  0.000879
ISINTOPFLOOR               0.000838
ISSTUDIO                   0.000139
CITY_Madrid                0.000069
CITY_Valencia              0.000034
dtype: float64


In [92]:
comparacion = pd.DataFrame({
    "Precio_real": y_test.values,
    "Precio_predicho": y_test_pred
})

print(comparacion.head(10))



   Precio_real  Precio_predicho
0     335000.0         324110.0
1     100000.0         114140.0
2     670000.0         688900.0
3    2486000.0        2611450.0
4      84000.0          80270.0
5     596000.0         583910.0
6     154000.0         146790.0
7     479000.0         467630.0
8     453000.0         417590.0
9     304000.0         275900.0


In [93]:
#Se probaron técnicas de mejora como la transformación logarítmica del target y la 
# regularización del modelo (limitando la complejidad de los árboles), pero ambas empeoraron el 
# rendimiento en el conjunto de test
# el modelo original ya se encontraba en un 
# equilibrio adecuado entre ajuste y generalización.
model2 = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

model2.fit(X_train, y_train)

y_train_pred = model2.predict(X_train)
y_test_pred = model2.predict(X_test)

print("=== TRAIN ===")
print(f"MAE:  {mean_absolute_error(y_train, y_train_pred):,.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred)):,.2f}")
print(f"R²:   {r2_score(y_train, y_train_pred):.4f}")

print("\n=== TEST ===")
print(f"MAE:  {mean_absolute_error(y_test, y_test_pred):,.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_test_pred)):,.2f}")
print(f"R²:   {r2_score(y_test, y_test_pred):.4f}")

=== TRAIN ===
MAE:  50,766.12
RMSE: 91,988.81
R²:   0.9316

=== TEST ===
MAE:  56,147.18
RMSE: 105,562.15
R²:   0.9097


In [100]:
from xgboost import XGBRegressor
model_xgb = XGBRegressor(
    n_estimators=800,
    max_depth=10,
    learning_rate=0.1,
    subsample=1.0,
    colsample_bytree=1.0,
    random_state=42,
    n_jobs=-1
)

model_xgb.fit(X_train, y_train)
y_train_pred_xgb = model_xgb.predict(X_train)
y_test_pred_xgb = model_xgb.predict(X_test)

print("=== TRAIN ===")
print("MAE:", mean_absolute_error(y_train, y_train_pred_xgb))
print("RMSE:", np.sqrt(mean_squared_error(y_train, y_train_pred_xgb)))
print("R²:", r2_score(y_train, y_train_pred_xgb))

print("\n=== TEST ===")
print("MAE:", mean_absolute_error(y_test, y_test_pred_xgb))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_test_pred_xgb)))
print("R²:", r2_score(y_test, y_test_pred_xgb))

=== TRAIN ===
MAE: 17720.793044297367
RMSE: 25657.81405482195
R²: 0.9946795994204476

=== TEST ===
MAE: 42903.531226964595
RMSE: 87466.59496244187
R²: 0.9380160723161799


In [ ]:
#AQUI MEJOR PO UNITPRICE --> por precio sigue siendo mejor

target = "UNITPRICE"

features = [
    "ROOMNUMBER",
    "BATHNUMBER",
    "HASTERRACE",
    "HASLIFT",
    "HASAIRCONDITIONING",
    "HASPARKINGSPACE",
    "HASBOXROOM",
    "HASWARDROBE",
    "HASSWIMMINGPOOL",
    "HASGARDEN",
    "ISDUPLEX",
    "ISSTUDIO",
    "ISINTOPFLOOR",
    "LATITUDE",
    "LONGITUDE",
    "DISTANCE_TO_CITY_CENTER",
    "DISTANCE_TO_METRO",
    "CITY"
]

df_unit = df[features + ["CONSTRUCTEDAREA", target]].copy()
df_unit = pd.get_dummies(df_unit, columns=["CITY"], drop_first=True)
df_unit = df_unit.dropna()
# Aquí CONSTRUCTEDAREA no va dentro de X si quieres que el modelo aprenda valor por m² sin apoyarse directamente en el tamaño.
X = df_unit.drop(columns=[target, "CONSTRUCTEDAREA"])
y = df_unit[target]

# Forzar nombres de columnas a str nativo de Python
X.columns = [str(col) for col in X.columns]
area = df_unit["CONSTRUCTEDAREA"]

X_train, X_test, y_train, y_test, area_train, area_test = train_test_split(
    X, y, area, test_size=0.2, random_state=42
)


#entrenar el modelo
model_unit = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model_unit.fit(X_train, y_train)

unitprice_pred_test = model_unit.predict(X_test)

price_pred_test = unitprice_pred_test * area_test
price_real_test = y_test * area_test

mae_price = mean_absolute_error(price_real_test, price_pred_test)
rmse_price = np.sqrt(mean_squared_error(price_real_test, price_pred_test))
r2_price = r2_score(price_real_test, price_pred_test)

print("=== PRICE reconstruido desde UNITPRICE ===")
print("MAE:", mae_price)
print("RMSE:", rmse_price)
print("R²:", r2_price)

=== PRICE reconstruido desde UNITPRICE ===
MAE: 47699.025957293976
RMSE: 92170.81870924604
R²: 0.9311694020002823
